<a href="https://colab.research.google.com/github/EstherWare/plp-da-a2-rainfall-advisory/blob/main/Copy_of_rainfall_analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import numpy as np

# Dataset provided in the assignment
counties = ["Nairobi", "Nakuru", "Kisumu", "Mombasa", "Machakos"]
clean_rows = [
    [64, 56, 92, 219, 176, 35, 18, 24, 31, 61, 165, 104],   # Nairobi
    [35, 42, 70, 145, 120, 78, 96, 105, 72, 55, 60, 38],    # Nakuru
    [58, 90, 150, 240, 180, 95, 70, 85, 90, 88, 130, 95],   # Kisumu
    [30, 15, 55, 150, 240, 110, 90, 65, 60, 90, 100, 70],   # Mombasa
]
machakos_raw = ["48", "35", "88", "N/A", "140", " 20 ",
                "9", "12", "", "45", "182", "96"]
month_names = ["Jan", "Feb", "Mar", "Apr", "May", "Jun",
               "Jul", "Aug", "Sep", "Oct", "Nov", "Dec"]

# Task 1: Clean Machakos raw data using try/except and impute missing values
def clean_machakos(raw_data, months):
    """Clean Machakos raw data, handle errors, and impute missing values with rounded average."""
    valid_nums = []
    for val in raw_data:
        try:
            valid_nums.append(int(val))
        except ValueError:
            pass

    avg_val = round(sum(valid_nums) / len(valid_nums))

    cleaned_row = []
    imputed_months = []

    for month, val in zip(months, raw_data):
        try:
            cleaned_row.append(int(val))
        except ValueError:
            cleaned_row.append(avg_val)
            imputed_months.append(month)

    print(f"Machakos Imputed Months: {imputed_months} (Imputed value: {avg_val} mm)")
    return cleaned_row

machakos_clean = clean_machakos(machakos_raw, month_names)

# Task 2: Combine all five counties into a single 5 x 12 NumPy array
all_rows = clean_rows + [machakos_clean]
rain_array = np.array(all_rows)

print(f"Array Shape: {rain_array.shape}")
print(f"Array Dtype: {rain_array.dtype}")

# Task 3: Annual total rainfall per county and average rainfall per month
annual_totals = rain_array.sum(axis=1)
monthly_averages = rain_array.mean(axis=0).round(1)

print("--- Annual Totals ---")
for county, total in zip(counties, annual_totals):
    print(f"{county:<10} total = {total} mm")

print("\n--- Monthly Averages (All Counties) ---")
for m_name, avg in zip(month_names, monthly_averages):
    print(f"{m_name}: {avg} mm")

# Task 4: Wettest month for each county using np.argmax with axis=1
wettest_indices = rain_array.argmax(axis=1)
wettest_months = [month_names[idx] for idx in wettest_indices]

print("\n--- Wettest Months ---")
for county, w_month in zip(counties, wettest_months):
    print(f"{county}: {w_month}")

# Task 5: Count dry months (< 50 mm) using a boolean mask
dry_months_counts = (rain_array < 50).sum(axis=1)

print("\n--- Dry Months Count (< 50 mm) ---")
for county, d_count in zip(counties, dry_months_counts):
    print(f"{county}: {d_count} months")

# Task 6: Identify recommended planting window (highest two consecutive months)
# Using shifted-array technique: R[:, :-1] + R[:, 1:]
consecutive_pairs = rain_array[:, :-1] + rain_array[:, 1:]
best_pair_indices = consecutive_pairs.argmax(axis=1)

planting_windows = []
for idx in best_pair_indices:
    m1 = month_names[idx]
    m2 = month_names[idx + 1]
    planting_windows.append(f"{m1}–{m2}")

print("\n--- Recommended Planting Windows ---")
for county, p_window in zip(counties, planting_windows):
    print(f"{county}: {p_window}")

Machakos Imputed Months: ['Apr', 'Sep'] (Imputed value: 68 mm)
Array Shape: (5, 12)
Array Dtype: int64
--- Annual Totals ---
Nairobi    total = 1045 mm
Nakuru     total = 916 mm
Kisumu     total = 1371 mm
Mombasa    total = 1075 mm
Machakos   total = 811 mm

--- Monthly Averages (All Counties) ---
Jan: 47.0 mm
Feb: 47.6 mm
Mar: 91.0 mm
Apr: 164.4 mm
May: 171.2 mm
Jun: 67.6 mm
Jul: 56.6 mm
Aug: 58.2 mm
Sep: 64.2 mm
Oct: 67.8 mm
Nov: 127.4 mm
Dec: 80.6 mm

--- Wettest Months ---
Nairobi: Apr
Nakuru: Apr
Kisumu: Apr
Mombasa: May
Machakos: Nov

--- Dry Months Count (< 50 mm) ---
Nairobi: 4 months
Nakuru: 3 months
Kisumu: 0 months
Mombasa: 2 months
Machakos: 6 months

--- Recommended Planting Windows ---
Nairobi: Apr–May
Nakuru: Apr–May
Kisumu: Apr–May
Mombasa: Apr–May
Machakos: Nov–Dec


In [2]:
from google.colab import drive
drive.mount('/content/drive')

MessageError: Error: credential propagation was unsuccessful

## Planting Advisory
* Kisumu, Nairobi, Nakuru, and Mombasa should prioritize their primary planting window during the April–May long rains, leveraging peak rainfall months (April for the western/central regions and May for Mombasa).
* Machakos operates on a distinct bimodal rainfall pattern and should focus its agricultural planting window during the November–December short rains, when seasonal precipitation peaks.
* Agricultural extension teams should deploy drought-advisory SMS campaigns to Coast and Machakos ahead of drier periods, as both counties experience significant dry months (6 and 2 months below 50 mm, respectively) that threaten crop yields.

## Data Quality Notes
* The raw Machakos dataset contained handwritten text errors, including missing values (`"N/A"` and empty strings `""`) and inconsistent spacing.
* These missing values were programmatically handled using a `try-except` block catching specific `ValueError` exceptions and successfully imputed using the rounded average of valid readings (68 mm).
* Analysts must note that the dataset covers a short window; long-term seasonal historical averages should be consulted before making large-scale agricultural investments.


In [3]:
import pandas as pd

# Create a structured DataFrame containing all your computed metrics
summary_df = pd.DataFrame({
    "County": counties,
    "Annual Total (mm)": annual_totals,
    "Wettest Month": wettest_months,
    "Dry Months (<50mm)": dry_months_counts,
    "Recommended Planting Window": planting_windows
})

# Display the DataFrame with clear formatting
display(summary_df)

# Export to CSV
summary_df.to_csv("rainfall_summary.csv", index=False)
print("\nSummary successfully exported to 'rainfall_summary.csv'!")

,County,Annual Total (mm),Wettest Month,Dry Months (<50mm),Recommended Planting Window
0,Nairobi,1045,Apr,4,Apr–May
1,Nakuru,916,Apr,3,Apr–May
2,Kisumu,1371,Apr,0,Apr–May
3,Mombasa,1075,May,2,Apr–May
4,Machakos,811,Nov,6,Nov–Dec



Summary successfully exported to 'rainfall_summary.csv'!


In [ ]:
import pandas as pd # or read with csv module to verify
import csv

with open("rainfall_summary.csv", mode="r", encoding="utf-8") as f:
    reader = csv.reader(f)
    for row in reader:
        print(row)

In [ ]:
%%writefile requirements.txt
numpy